# Budget & Demand Analysis
**Keandra Morris | Operations Project Manager**  
Portfolio: [keandrajm.github.io/PM_Portfolio](https://keandrajm.github.io/PM_Portfolio/)  

---
This notebook demonstrates budget variance analysis and demand forecasting for an operations program — the same analytical work applied to a $500K annual budget across a 12,000+ user program.

**Analysis covers:**
- Budget vs. actual variance by month
- Waste reduction trend and cost savings
- Demand (user volume) forecasting using linear regression
- Cost-per-user efficiency metric

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

GREEN = '#1B4332'
GOLD  = '#B8933A'
RED   = '#C0392B'
TEAL  = '#2980B9'
MUTED = '#4A5568'
BG    = '#F7F6F1'

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

df = pd.DataFrame({
    'Month':         months,
    'Daily_Users':   [285,290,310,305,320,315,298,312,328,335,342,338],
    'Budget_Spent':  [38500,39200,41000,40500,42000,41800,39900,41200,43000,44100,45000,44500],
    'Budget_Target': [41000]*12,
    'Waste_Pct':     [8.2,7.8,7.1,6.8,6.2,5.9,5.7,5.4,5.1,4.8,4.5,4.3],
})

df['Budget_Variance']  = df['Budget_Target'] - df['Budget_Spent']
df['Cost_Per_User']    = (df['Budget_Spent'] / (df['Daily_Users'] * 30)).round(2)
df['Waste_Saved_$']    = ((df['Waste_Pct'].iloc[0] - df['Waste_Pct']) / 100 * df['Budget_Spent']).round(0)

df[['Month','Budget_Spent','Budget_Variance','Cost_Per_User','Waste_Saved_$']].head(6)

## 2. Budget Variance Analysis

In [ ]:
total_savings  = df['Budget_Variance'].sum()
over_budget    = (df['Budget_Variance'] < 0).sum()
under_budget   = (df['Budget_Variance'] >= 0).sum()
avg_variance   = df['Budget_Variance'].mean()
best_month     = df.loc[df['Budget_Variance'].idxmax(), 'Month']

print('BUDGET VARIANCE SUMMARY')
print('─' * 35)
print(f'  Total budget savings:    ${total_savings:,.0f}')
print(f'  Months under budget:     {under_budget}/12')
print(f'  Months over budget:      {over_budget}/12')
print(f'  Avg monthly variance:    ${avg_variance:,.0f}')
print(f'  Best performing month:   {best_month}')
print(f'  Annualized cost/user:    ${df["Cost_Per_User"].mean()*12:.2f}')

## 3. Demand Forecasting (Linear Regression)

In [ ]:
x = np.arange(12)
slope, intercept, r, p, se = stats.linregress(x, df['Daily_Users'])

# Project next 3 months
future_x    = np.arange(12, 15)
future_pred = slope * future_x + intercept
future_months = ['Jan+1','Feb+1','Mar+1']

print('DEMAND FORECAST — Next 3 Months')
print('─' * 35)
for m, p_val in zip(future_months, future_pred):
    print(f'  {m}:  ~{p_val:.0f} users/day (projected)')
print(f'\n  Trend slope:  +{slope:.1f} users/month')
print(f'  R² value:     {r**2:.3f}  (model fit)')

fig, ax = plt.subplots(figsize=(11, 4.5), facecolor=BG)
ax.set_facecolor('white')
ax.fill_between(x, df['Daily_Users'], alpha=0.1, color=TEAL)
ax.plot(x, df['Daily_Users'], color=TEAL, linewidth=2.5, marker='s',
        markersize=5, markerfacecolor='white', markeredgecolor=TEAL, markeredgewidth=2,
        label='Actual')
trend_line = slope * np.arange(15) + intercept
ax.plot(range(15), trend_line, '--', color=MUTED, linewidth=1.2, alpha=0.6, label='Trend line')
ax.plot(future_x, future_pred, color=GOLD, linewidth=2.5, marker='D',
        markersize=6, markerfacecolor=GOLD, linestyle='--', label='Forecast')
ax.axvline(11.5, color='#CBD5E0', linewidth=1.5, linestyle=':')
ax.text(11.6, 295, 'Forecast →', fontsize=8.5, color=MUTED)
ax.set_xticks(list(range(12)) + list(future_x))
ax.set_xticklabels(months + future_months, fontsize=8)
ax.set_ylabel('Users / Day')
ax.set_title('Daily User Demand — Actual vs. Forecast', fontsize=12, fontweight='bold', color=GREEN)
ax.legend(fontsize=9, frameon=False)
ax.grid(axis='y', alpha=0.3)
for s in ['top','right']: ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Cost-Per-User Efficiency Trend

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), facecolor=BG)

# Cost per user
ax1.set_facecolor('white')
bar_colors = [RED if c > df['Cost_Per_User'].mean() else GREEN for c in df['Cost_Per_User']]
ax1.bar(range(12), df['Cost_Per_User'], color=bar_colors, alpha=0.82, width=0.6)
ax1.axhline(df['Cost_Per_User'].mean(), color=GOLD, linestyle='--', linewidth=1.5,
            label=f'Avg: ${df["Cost_Per_User"].mean():.2f}')
ax1.set_xticks(range(12)); ax1.set_xticklabels(months, fontsize=8)
ax1.set_ylabel('$/user/day'); ax1.set_title('Cost Per User Per Day', fontweight='bold', color=GREEN)
ax1.legend(fontsize=9, frameon=False)
ax1.grid(axis='y', alpha=0.3)
for s in ['top','right']: ax1.spines[s].set_visible(False)

# Waste savings
ax2.set_facecolor('white')
cumulative_savings = df['Waste_Saved_$'].cumsum()
ax2.fill_between(range(12), cumulative_savings, alpha=0.15, color=GREEN)
ax2.plot(range(12), cumulative_savings, color=GREEN, linewidth=2.5, marker='o',
         markersize=5, markerfacecolor=GOLD, markeredgecolor=GREEN, markeredgewidth=2)
ax2.set_xticks(range(12)); ax2.set_xticklabels(months, fontsize=8)
ax2.set_ylabel('Cumulative Savings ($)')
ax2.set_title('Cumulative Waste Reduction Savings', fontweight='bold', color=GREEN)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax2.grid(axis='y', alpha=0.3)
for s in ['top','right']: ax2.spines[s].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Total waste reduction savings: ${df['Waste_Saved_$'].sum():,.0f}")

---
**Notebook author:** Keandra Morris  
**Portfolio:** [keandrajm.github.io/PM_Portfolio](https://keandrajm.github.io/PM_Portfolio/)  
**Tools used:** Python · Pandas · NumPy · Matplotlib · SciPy